<a href="https://colab.research.google.com/github/NyssaRomana/IDCamp-Dicoding/blob/main/LatihanSentimenAnalisis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install google-play-scraper
# Mengimpor pustaka google_play_scraper untuk mengakses ulasan dan informasi aplikasi dari Google Play Store.
from google_play_scraper import app, reviews, Sort, reviews_all

import pandas as pd # Pandas untuk manipulasi dan analisis data
pd.options.mode.chained_assignment = None # Menonaktifkan peringatan chaining
import numpy as np # NumPy untuk komputasi numerik
seed = 0
np.random.seed(seed) # Mengatur seed untuk reproduktibilitas
import matplotlib.pyplot as plt # Matplotlib untuk visualisasi data
import seaborn as sns # Seaborn untuk visualisasi data statistik, mengatur gaya visualisasi
from sklearn.metrics import accuracy_score

import datetime as dt # Manipulasi data waktu dan tanggal
import re # Modul untuk bekerja dengan ekspresi reguler
import string # Berisi konstanta string, seperti tanda baca
from nltk.tokenize import word_tokenize # Tokenisasi teks
from nltk.corpus import stopwords # Daftar kata-kata berhenti dalam teks

!pip install sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory # Stemming (penghilangan imbuhan kata) dalam bahasa Indonesia
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory # Menghapus kata-kata berhenti dalam bahasa Indonesia

from wordcloud import WordCloud # Membuat visualisasi berbentuk awan kata (word cloud) dari teks

import nltk # Import pustaka NLTK (Natural Language Toolkit).
nltk.download('punkt') # Mengunduh dataset yang diperlukan untuk tokenisasi teks.
nltk.download('stopwords') # Mengunduh dataset yang berisi daftar kata-kata berhenti (stopwords) dalam berbagai bahasa.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 10.2 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
# Mengimpor pustaka google_play_scraper untuk mengakses ulasan dan informasi aplikasi dari Google Play Store.
from google_play_scraper import app, reviews_all, Sort

# Mengambil semua ulasan dari aplikasi dengan ID 'com.byu.id' di Google Play Store.
# Proses scraping mungkin memerlukan beberapa saat tergantung pada jumlah ulasan yang ada.
scrapreview = reviews_all(
    'com.byu.id',         # ID aplikasi
    lang='id',            # Bahasa ulasan (default: 'en')
    country='id',         # Negara (default: 'us')
    sort=Sort.MOST_RELEVANT, # Urutan ulasan (default: Sort.MOST_RELEVANT)
    count=1000            # Jumlah maksimum ulasan yang ingin diambil
)

In [4]:
# Menyimpan ulasan dalam file CSV
import csv

with open('ulasan_aplikasi.csv', mode='w', newline='', encoding='utf-8') as file:
  writer = csv.writer(file)
  writer.writerow(['Review']) # Menulis header kolom
  for review in scrapreview:
    writer.writerow([review['content']]) # Menulis konten ulasan ke dalam file CSV

In [5]:
app_reviews_df = pd.DataFrame(scrapreview)
app_reviews_df.shape
app_reviews_df.head()
app_reviews_df.to_csv('ulasan_aplikasi.csv', index=False)

# Membuat DataFrame dari hasil scrapreview
app_reviews_df = pd.DataFrame(scrapreview)

# Menghitung jumlah baris dan kolom dalam DataFrame
jumlah_ulasan, jumlah_kolom = app_reviews_df.shape

In [6]:
# Menampilkan lima baris pertama dari DataFrame app_reviews_df
app_reviews_df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,58f237aa-4b5d-4315-9650-6f348f715234,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Gw pengguna by.u dari lama, awal awal muncul. ...",1,521,1.64.1-prod-byu,2026-01-19 11:31:39,"Hi Kak Fajar, maaf belum bisa bikin kamu nyama...",2026-01-19 11:36:14,1.64.1-prod-byu
1,43d091d6-4047-48aa-84b0-9391605c06c7,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Banyak error! aku udh cape dengan byu karena s...,1,68,1.64.1-prod-byu,2026-01-18 07:52:20,"Hai Kak, maaf ya udah bikin ga nyaman :( Terka...",2026-01-18 07:54:12,1.64.1-prod-byu
2,b99b3a04-fed5-4b33-a891-ea7d9f7402e4,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Setelah update, tampilan usernya jadi tidak us...",1,47,1.64.1-prod-byu,2026-01-21 08:43:55,Hallo Kak Joshua. Maaf udah bikin gak nyaman. ...,2026-01-21 08:44:18,1.64.1-prod-byu
3,0c65f840-0a78-4cb3-8d28-48b75afb744f,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Saya user lama sejak by.U launching. Aplikasin...,2,184,1.64.1-prod-byu,2026-01-06 10:21:07,"Hai, Kak PRAM. Maafin udah bikin Kakak jadi ga...",2026-01-06 10:44:06,1.64.1-prod-byu
4,a24da47c-aed6-4bf1-a6c6-1c4f82f139de,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,aduhhh kok fitur(atur semaunya)di hilangkan in...,1,334,1.64.1-prod-byu,2026-01-11 03:15:45,"Hai Kak, maaf udah bikin ga nyaman ya :( Kalau...",2026-01-11 06:51:49,1.64.1-prod-byu


In [7]:
# Menampilkan informasi tentang DataFrame app_reviews_df
app_reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208809 entries, 0 to 208808
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   reviewId              208809 non-null  object        
 1   userName              208809 non-null  object        
 2   userImage             208809 non-null  object        
 3   content               208808 non-null  object        
 4   score                 208809 non-null  int64         
 5   thumbsUpCount         208809 non-null  int64         
 6   reviewCreatedVersion  175488 non-null  object        
 7   at                    208809 non-null  datetime64[ns]
 8   replyContent          187933 non-null  object        
 9   repliedAt             187933 non-null  datetime64[ns]
 10  appVersion            175488 non-null  object        
dtypes: datetime64[ns](2), int64(2), object(7)
memory usage: 17.5+ MB


In [8]:
# Membuat DataFrame baru (clean_df) dengan menghapus baris yang memiliki nilai yang hilang (NaN) dari app_reviews_df
clean_df = app_reviews_df.dropna()

In [9]:
# Menghapus baris duplikat dari DataFrame clean_df
clean_df = clean_df.drop_duplicates()

# Menghitung jumlah baris dan kolom dalam DataFrame clean_df setelah menghapus duplikat
jumlah_ulasan_setelah_hapus_duplikat, jumlah_kolom_setelah_hapus_duplikat = clean_df.shape

In [10]:
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 157303 entries, 0 to 200573
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   reviewId              157303 non-null  object        
 1   userName              157303 non-null  object        
 2   userImage             157303 non-null  object        
 3   content               157303 non-null  object        
 4   score                 157303 non-null  int64         
 5   thumbsUpCount         157303 non-null  int64         
 6   reviewCreatedVersion  157303 non-null  object        
 7   at                    157303 non-null  datetime64[ns]
 8   replyContent          157303 non-null  object        
 9   repliedAt             157303 non-null  datetime64[ns]
 10  appVersion            157303 non-null  object        
dtypes: datetime64[ns](2), int64(2), object(7)
memory usage: 14.4+ MB


In [11]:
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) # menghapus link
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka

    text = text.replace('\n', ' ') # mengganti baris baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
    text = text.strip(' ') # menghapus karakter spasi dari kiri dan kanan teks
    return text

def casefoldingText(text): # Mengubah semua karakter dalam teks menjadi huruf kecil
    text = text.lower()
    return text

def tokenizingText(text): # Memecah atau membagi string, teks menjadi daftar token
    text = word_tokenize(text)
    return text

def filteringText(text): # Menghapus stopwords dalam teks
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords1 = set(stopwords.words('english'))
    listStopwords.update(listStopwords1)
    listStopwords.update(['iya','yaa','gak','nya','na','sih','ku',"di","ga","ya","gaa","loh","kah","woi","woii","woy"])
    filtered = []
    for txt in text:
      if txt not in listStopwords:
          filtered.append(txt)
    text = filtered
    return text

def stemmingText(text): # Mengurangi kata ke bentuk dasarnya yang menghilangkan imbuhan awalan dan akhiran atau ke akar kata
    # Membuat objek stemmer
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()

    # Memecah teks menjadi daftar kata
    words = text.split()

    # Menerapkan stemming pada setiap kata dalam daftar
    stemmed_words = [stemmer.stem(word) for word in words]

    # Menggabungkan kata-kata yang telah distem
    stemmed_text = ' '.join(stemmed_words)

    return stemmed_text

def toSentence(list_words): # Mengubah daftar kata menjadi kalimat
    sentence = ' '.join(word for word in list_words)
    return sentence

In [23]:
slangwords = {
    "@": "di",
    "abis": "habis",
    "wtb": "beli",
    "masi": "masih",
    "wts": "jual",
    "wtt": "tukar",
    "bgt": "banget",
    "maks": "maksimal",
    "btw": "ngomong-ngomong",
    "gws": "lekas sembuh",
    "otw": "sedang dalam perjalanan",
    "pap": "kirim foto",
    "sotoy": "sok tahu",
    "gw": "saya",
    "udah": "sudah",
    "gk": "tidak",
    "udh": "sudah",
    "ga": "tidak"
}
def fix_slangwords(text):
    words = text.split()
    fixed_words = []

    for word in words:
        if word.lower() in slangwords:
            fixed_words.append(slangwords[word.lower()])
        else:
            fixed_words.append(word)

    fixed_text = ' '.join(fixed_words)
    return fixed_text

In [24]:
import nltk
nltk.download('punkt_tab')

# Membersihkan teks dan menyimpannya di kolom 'text_clean'
clean_df['text_clean'] = clean_df['content'].apply(cleaningText)

# Mengubah huruf dalam teks menjadi huruf kecil dan menyimpannya di 'text_casefoldingText'
clean_df['text_casefoldingText'] = clean_df['text_clean'].apply(casefoldingText)

# Mengganti kata-kata slang dengan kata-kata standar dan menyimpannya di 'text_slangwords'
clean_df['text_slangwords'] = clean_df['text_casefoldingText'].apply(fix_slangwords)

# Memecah teks menjadi token (kata-kata) dan menyimpannya di 'text_tokenizingText'
clean_df['text_tokenizingText'] = clean_df['text_slangwords'].apply(tokenizingText)

# Menghapus kata-kata stop (kata-kata umum) dan menyimpannya di 'text_stopword'
clean_df['text_stopword'] = clean_df['text_tokenizingText'].apply(filteringText)

# Menggabungkan token-token menjadi kalimat dan menyimpannya di 'text_akhir'
clean_df['text_akhir'] = clean_df['text_stopword'].apply(toSentence)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [25]:
clean_df

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,text_clean,text_casefoldingText,text_slangwords,text_tokenizingText,text_stopword,text_akhir
0,58f237aa-4b5d-4315-9650-6f348f715234,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Gw pengguna by.u dari lama, awal awal muncul. ...",1,521,1.64.1-prod-byu,2026-01-19 11:31:39,"Hi Kak Fajar, maaf belum bisa bikin kamu nyama...",2026-01-19 11:36:14,1.64.1-prod-byu,Gw pengguna byu dari lama awal awal muncul Gw ...,gw pengguna byu dari lama awal awal muncul gw ...,saya pengguna byu dari lama awal awal muncul s...,"[saya, pengguna, byu, dari, lama, awal, awal, ...","[pengguna, byu, muncul, jujur, nyaman, pake, b...",pengguna byu muncul jujur nyaman pake byu kepa...
1,43d091d6-4047-48aa-84b0-9391605c06c7,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Banyak error! aku udh cape dengan byu karena s...,1,68,1.64.1-prod-byu,2026-01-18 07:52:20,"Hai Kak, maaf ya udah bikin ga nyaman :( Terka...",2026-01-18 07:54:12,1.64.1-prod-byu,Banyak error aku udh cape dengan byu karena se...,banyak error aku udh cape dengan byu karena se...,banyak error aku sudah cape dengan byu karena ...,"[banyak, error, aku, sudah, cape, dengan, byu,...","[error, cape, byu, buka, uninstall, login, err...",error cape byu buka uninstall login error beli...
2,b99b3a04-fed5-4b33-a891-ea7d9f7402e4,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Setelah update, tampilan usernya jadi tidak us...",1,47,1.64.1-prod-byu,2026-01-21 08:43:55,Hallo Kak Joshua. Maaf udah bikin gak nyaman. ...,2026-01-21 08:44:18,1.64.1-prod-byu,Setelah update tampilan usernya jadi tidak use...,setelah update tampilan usernya jadi tidak use...,setelah update tampilan usernya jadi tidak use...,"[setelah, update, tampilan, usernya, jadi, tid...","[update, tampilan, usernya, user, friendly, ha...",update tampilan usernya user friendly halaman ...
3,0c65f840-0a78-4cb3-8d28-48b75afb744f,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Saya user lama sejak by.U launching. Aplikasin...,2,184,1.64.1-prod-byu,2026-01-06 10:21:07,"Hai, Kak PRAM. Maafin udah bikin Kakak jadi ga...",2026-01-06 10:44:06,1.64.1-prod-byu,Saya user lama sejak byU launching Aplikasinya...,saya user lama sejak byu launching aplikasinya...,saya user lama sejak byu launching aplikasinya...,"[saya, user, lama, sejak, byu, launching, apli...","[user, byu, launching, aplikasinya, rapi, siny...",user byu launching aplikasinya rapi sinyal oke...
4,a24da47c-aed6-4bf1-a6c6-1c4f82f139de,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,aduhhh kok fitur(atur semaunya)di hilangkan in...,1,334,1.64.1-prod-byu,2026-01-11 03:15:45,"Hai Kak, maaf udah bikin ga nyaman ya :( Kalau...",2026-01-11 06:51:49,1.64.1-prod-byu,aduhhh kok fituratur semaunyadi hilangkan ini ...,aduhhh kok fituratur semaunyadi hilangkan ini ...,aduhhh kok fituratur semaunyadi hilangkan ini ...,"[aduhhh, kok, fituratur, semaunyadi, hilangkan...","[aduhhh, fituratur, semaunyadi, hilangkan, kal...",aduhhh fituratur semaunyadi hilangkan kalo isi...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200564,cd5f5eaf-f7fa-4619-b4ec-4f8f627d0cbe,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Oke,5,0,1.0.318 - by.U,2019-10-25 12:44:20,Hi kak makasi yah buat support nya :),2019-10-28 02:42:49,1.0.318 - by.U,Oke,oke,oke,[oke],[oke],oke
200568,5d3c977c-f441-4278-be52-69d670684a58,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Ga bsa dafatar via fb,1,1,1.0.313,2019-10-19 09:02:35,"hai hai, untuk troubleshoot solution bisa hubu...",2019-10-19 11:22:14,1.0.313,Ga bsa dafatar via fb,ga bsa dafatar via fb,tidak bsa dafatar via fb,"[tidak, bsa, dafatar, via, fb]","[bsa, dafatar, via, fb]",bsa dafatar via fb
200569,a81ed629-6cc7-4a68-8d9f-fe527924dab2,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Gak bisa sign up ata

In [26]:
import csv
import requests
from io import StringIO

# Membaca data kamus kata-kata positif dari GitHub
lexicon_positive = dict()

response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv')
# Mengirim permintaan HTTP untuk mendapatkan file CSV dari GitHub

if response.status_code == 200:
    # Jika permintaan berhasil
    reader = csv.reader(StringIO(response.text), delimiter=',')
    # Membaca teks respons sebagai file CSV menggunakan pembaca CSV dengan pemisah koma

    for row in reader:
        # Mengulangi setiap baris dalam file CSV
        lexicon_positive[row[0]] = int(row[1])
        # Menambahkan kata-kata positif dan skornya ke dalam kamus lexicon_positive
else:
    print("Failed to fetch positive lexicon data")

# Membaca data kamus kata-kata negatif dari GitHub
lexicon_negative = dict()

response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv')
# Mengirim permintaan HTTP untuk mendapatkan file CSV dari GitHub

if response.status_code == 200:
    # Jika permintaan berhasil
    reader = csv.reader(StringIO(response.text), delimiter=',')
    # Membaca teks respons sebagai file CSV menggunakan pembaca CSV dengan pemisah koma

    for row in reader:
        # Mengulangi setiap baris dalam file CSV
        lexicon_negative[row[0]] = int(row[1])
        # Menambahkan kata-kata negatif dan skornya dalam kamus lexicon_negative
else:
    print("Failed to fetch negative lexicon data")

In [27]:
# Fungsi untuk menentukan polaritas sentimen dari tweet

def sentiment_analysis_lexicon_indonesia(text):
    #for word in text:

    score = 0
    # Inisialisasi skor sentimen ke 0

    for word in text:
        # Mengulangi setiap kata dalam teks

        if (word in lexicon_positive):
            score = score + lexicon_positive[word]
            # Jika kata ada dalam kamus positif, tambahkan skornya ke skor sentimen

    for word in text:
        # Mengulangi setiap kata dalam teks (sekali lagi)

        if (word in lexicon_negative):
            score = score + lexicon_negative[word]
            # Jika kata ada dalam kamus negatif, kurangkan skornya dari skor sentimen

    polarity=''
    # Inisialisasi variabel polaritas

    if (score >= 0):
        polarity = 'positive'
        # Jika skor sentimen lebih besar atau sama dengan 0, maka polaritas adalah positif
    elif (score < 0):
          polarity = 'negative'
          # Jika skor sentimen kurang dari 0, maka polaritas adalah negatif

    # else:
    #     polarity = 'neutral'
    # Ini adalah bagian yang bisa digunakan untuk menentukan polaritas netral jika diperlukan

    return score, polarity
    # Mengembalikan skor sentimen dan polaritas teks

In [36]:
results = clean_df['text_stopword'].apply(sentiment_analysis_lexicon_indonesia)
results = list(zip(*results))
clean_df['polarity_score'] = results[0]
clean_df['polarity'] = results[1]
print(clean_df['polarity'].value_counts())

polarity
positive    91173
negative    66130
Name: count, dtype: int64


In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Pisahkan data menjadi fitur (tweet) dan label (sentimen)
X = clean_df['text_akhir']
y = clean_df['polarity']

# Ekstraksi fitur dengan TF-IDF
tfidf = TfidfVectorizer(max_features=200, min_df=17, max_df=0.8 )
X_tfidf = tfidf.fit_transform(X)

# Konversi hasil ekstraksi fitur menjadi dataframe
features_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())

# Menampilkan hasil ekstraksi fitur
features_df

# Bagi data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

**Naive Bayes**

In [39]:
from sklearn.naive_bayes import BernoulliNB

# Membuat objek model Naive Bayes (Bernoulli Naive Bayes)
naive_bayes = BernoulliNB()

# Melatih model Naive Bayes pada data pelatihan
naive_bayes.fit(X_train.toarray(), y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_nb = naive_bayes.predict(X_train.toarray())
y_pred_test_nb = naive_bayes.predict(X_test.toarray())

# Evaluasi akurasi model Naive Bayes
accuracy_train_nb = accuracy_score(y_pred_train_nb, y_train)
accuracy_test_nb = accuracy_score(y_pred_test_nb, y_test)

# Menampilkan akurasi
print('Naive Bayes - accuracy_train:', accuracy_train_nb)
print('Naive Bayes - accuracy_test:', accuracy_test_nb)

Naive Bayes - accuracy_train: 0.7922792072598973
Naive Bayes - accuracy_test: 0.7880868376720384


In [40]:
from sklearn.ensemble import RandomForestClassifier

# Membuat objek model Random Forest
random_forest = RandomForestClassifier()

# Melatih model Random Forest pada data pelatihan
random_forest.fit(X_train.toarray(), y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_rf = random_forest.predict(X_train.toarray())
y_pred_test_rf = random_forest.predict(X_test.toarray())

# Evaluasi akurasi model Random Forest
accuracy_train_rf = accuracy_score(y_pred_train_rf, y_train)
accuracy_test_rf = accuracy_score(y_pred_test_rf, y_test)

# Menampilkan akurasi
print('Random Forest - accuracy_train:', accuracy_train_rf)
print('Random Forest - accuracy_test:', accuracy_test_rf)

Random Forest - accuracy_train: 0.9705424262170023
Random Forest - accuracy_test: 0.9013699500969454


In [41]:
from sklearn.linear_model import LogisticRegression

# Membuat objek model Logistic Regression
logistic_regression = LogisticRegression()

# Melatih model Logistic Regression pada data pelatihan
logistic_regression.fit(X_train.toarray(), y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_lr = logistic_regression.predict(X_train.toarray())
y_pred_test_lr = logistic_regression.predict(X_test.toarray())

# Evaluasi akurasi model Logistic Regression pada data pelatihan
accuracy_train_lr = accuracy_score(y_pred_train_lr, y_train)

# Evaluasi akurasi model Logistic Regression pada data uji
accuracy_test_lr = accuracy_score(y_pred_test_lr, y_test)

# Menampilkan akurasi
print('Logistic Regression - accuracy_train:', accuracy_train_lr)
print('Logistic Regression - accuracy_test:', accuracy_test_lr)

Logistic Regression - accuracy_train: 0.9118497798827101
Logistic Regression - accuracy_test: 0.9125266202600044


In [42]:
from sklearn.tree import DecisionTreeClassifier

# Membuat objek model Decision Tree
decision_tree = DecisionTreeClassifier()

# Melatih model Decision Tree pada data pelatihan
decision_tree.fit(X_train.toarray(), y_train)

# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_dt = decision_tree.predict(X_train.toarray())
y_pred_test_dt = decision_tree.predict(X_test.toarray())

# Evaluasi akurasi model Decision Tree
accuracy_train_dt = accuracy_score(y_pred_train_dt, y_train)
accuracy_test_dt = accuracy_score(y_pred_test_dt, y_test)

# Menampilkan akurasi
print('Decision Tree - accuracy_train:', accuracy_train_dt)
print('Decision Tree - accuracy_test:', accuracy_test_dt)

Decision Tree - accuracy_train: 0.9705503726895631
Decision Tree - accuracy_test: 0.873939162772957
